In [ ]:
import torch

# This gives us a nice way of summarising our model layers and parameters
!pip install torchinfo

print(torch.__version__)

# Set some random seeds so that we should get the same answer!
torch.manual_seed(42)
torch.use_deterministic_algorithms(True)

# Define all of the random tensors for the unit tests to ensure we have the same
# numbers!
unit_test_1_input = torch.rand(1, 5, 4, dtype=torch.float32)
unit_test_2_input = torch.rand(1, 5, 4, dtype=torch.float32)
unit_test_3_input = torch.rand(1, 5, 4, dtype=torch.float32)
unit_test_4_input = torch.rand(1, 5, 4, dtype=torch.float32)

import os
import urllib.request
import zipfile

url = "http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"

filename = "spa-eng.zip"

if not os.path.exists(filename):
    urllib.request.urlretrieve(url, filename)

with zipfile.ZipFile(filename, "r") as zip_ref:
    zip_ref.extractall(".")

import pathlib
data_path = "spa-eng/spa.txt"

In [ ]:
with open(data_path) as data_file:
    lines = data_file.read().split("\n")[:-1]

import re
# Now convert the lines into English to Spanish pairs
# Since this is a simple model, it makes sense to enforce lower case and
# remove any punctuation
english_spanish_pairs = []
max_length_english = 0
for line in lines:
    line = line.lower()
    # Below regex removes characters not in the list, don't forget tabs (\t)!
    # We need to make sure that we keep the special characters from Spanish
    line = re.sub(r'[^A-Za-z0-9 \táéíóúñ]+', '', line)
    english, spanish = line.split("\t")
    # As seen in the lecture, we need to add start and end tokens for Spanish
    spanish = "<sos> " + spanish + " <eos>"
    english_spanish_pairs.append((english, spanish))


In [ ]:
import random

random.seed(42)
# Shuffle the dataset and split into training and validation samples (80%/20%)
random.shuffle(english_spanish_pairs)
# Let's have a look at a couple of examples
print('We have', len(english_spanish_pairs), 'examples in the dataset')
for w in range (5):
    print('Example',w,':', english_spanish_pairs[w])

num_training = int(0.8*len(english_spanish_pairs))
training_pairs = english_spanish_pairs[:num_training]
validation_pairs = english_spanish_pairs[num_training:]

print("Training sample size =", len(training_pairs),
      "and validation sample =", len(validation_pairs))

In [ ]:
from collections import Counter

# Define the maximum vocabulary size that we want to use. If there
# are more words than this, then the 15000 most frequent words are
# used as the vocabulary
vocab_size_english = 15000
vocab_size_spanish = 15000

sequence_length = 20

# Function to create the vocabulary
def build_vocab(sentences, max_tokens):
    counter = Counter()

    for sentence in sentences:
        words = sentence.split()
        counter.update(words)

    # Reserve 0 for padding and 1 for unknown
    vocab = {"<pad>": 0, "<unk>": 1}

    # Most common words fill the rest
    for word, _ in counter.most_common(max_tokens - 2):
        vocab[word] = len(vocab)

    return vocab


english_words = []
spanish_words = []

for pair in english_spanish_pairs:
    english_words.append(pair[0])
    spanish_words.append(pair[1])

english_vocab = build_vocab(english_words, vocab_size_english)
spanish_vocab = build_vocab(spanish_words, vocab_size_spanish)

def vectorise_sentence(sentence, vocab, max_length):

    tokens = sentence.split()

    ids = [
        vocab.get(token, vocab["<unk>"])
        for token in tokens
    ]

    # Truncate
    ids = ids[:max_length]

    # Pad
    ids += [vocab["<pad>"]] * (max_length - len(ids))

    return ids

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

# Create the dataset to store our English-Spanish sentence pairs
class TranslationDataset(Dataset):
    def __init__(
        self,
        pairs,
        english_vocab,
        spanish_vocab,
        sequence_length
    ):
        self.pairs = pairs
        self.english_vocab = english_vocab
        self.spanish_vocab = spanish_vocab
        self.sequence_length = sequence_length

    def __len__(self):
        return len(self.pairs)

    def vectorise_sentence(self, sentence, vocab, max_length):

        tokens = sentence.split()

        ids = [
            vocab.get(token, vocab["<unk>"])
            for token in tokens
        ]

        # truncate
        ids = ids[:max_length]

        # pad
        ids += [
            vocab["<pad>"]
        ] * (max_length - len(ids))

        return ids

    def __getitem__(self, idx):

        english, spanish = self.pairs[idx]

        # Remove <eos> for decoder input
        spanish_input = spanish.rsplit(" ", 1)[0]

        # Remove <sos> for target
        spanish_target = spanish.split(" ", 1)[1]

        encoder_inputs = self.vectorise_sentence(
            english,
            self.english_vocab,
            self.sequence_length
        )

        decoder_inputs = self.vectorise_sentence(
            spanish_input,
            self.spanish_vocab,
            self.sequence_length + 1
        )

        target = self.vectorise_sentence(
            spanish_target,
            self.spanish_vocab,
            self.sequence_length + 1
        )

        return (
            torch.tensor(encoder_inputs, dtype=torch.long),
            torch.tensor(decoder_inputs, dtype=torch.long),
            torch.tensor(target, dtype=torch.long)
        )

# We have to choose the batch size before creating the
# data loader objects below
batch_size = 64

training_dataset = TranslationDataset(
    training_pairs,
    english_vocab,
    spanish_vocab,
    sequence_length
)

validation_dataset = TranslationDataset(
    validation_pairs,
    english_vocab,
    spanish_vocab,
    sequence_length
)

training_dataloader = DataLoader(
    training_dataset,
    batch_size=batch_size,
    shuffle=True
)

validation_dataloader = DataLoader(
    validation_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
encoder_inputs, decoder_inputs, targets = next(
    iter(training_dataloader)
)

print(
    "inputs['encoder_inputs'] shape: ",
    encoder_inputs.shape
)

print(
    "inputs['decoder_inputs'] shape: ",
    decoder_inputs.shape
)

print(
    "targets shape: ",
    targets.shape
)

print(encoder_inputs[0])
print(decoder_inputs[0])
print(targets[0])

---

We have (finally) formatted the training and validation data into a format we can use. Now let's get on with defining all of the components that we need for our transformer.
1. Word embedding and position encoding
2. Multi-head attention
3. Feed-forward network
4. The encoder
5. The decoder
6. The transformer (brings together all of the above)

---

The following function is just included for completeness. It is a vectorised way to calculated the sinusoidal position encoding that we saw in the lectures:

$PE(j, 2k) = \sin\left(\frac{j}{10000^{\frac{2k}{d_{\textrm{model}}}}}\right), ~~ PE(j, 2k+1) = \cos\left(\frac{j}{10000^{\frac{2k}{d_{\textrm{model}}}}}\right)$

Our transformer will actually use a learned position encoding.

In [ ]:
import torch
import torch.nn as nn

class SinusoidalPositionEncoding(nn.Module):
    def __init__(self, sequence_length, d_model):
        super().__init__()

        position = torch.arange(sequence_length).unsqueeze(1)

        div_term = torch.exp(
            -torch.log(torch.tensor(10000.0))
            * (torch.arange(0, d_model, 2) / d_model)
        )

        pe = torch.zeros(sequence_length, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe)

    def forward(self, inputs=None):
        return self.pe

This class performs both the word embedding and the position encoding. We just make use of the `torch.nn.Embedding` layer here for simplicity. In our case, when using the learned position encoding we are using the same type of `torch` layer.

In [ ]:
import torch
import torch.nn as nn

class EmbedAndEncode(nn.Module):
    def __init__(self, sequence_length, d_model, vocab_size, learned_encoding):
        super().__init__()

        self.l = sequence_length

        # Token embeddings
        self.embedding = nn.Embedding(vocab_size, d_model)

        if learned_encoding:
            self.position = nn.Embedding(sequence_length, d_model)
        else:
            self.position = SinusoidalPositionEncoding(sequence_length, d_model)

    def forward(self, inputs):
        # inputs shape: (batch_size, seq_len)

        e = self.embedding(inputs)
        # shape: (batch_size, seq_len, d_model)

        pos = torch.arange(0, self.l, device=inputs.device, dtype=torch.long)

        p = self.position(pos)
        # shape: (seq_len, d_model)

        x = e + p
        # Broadcasting over batch dimension

        return x

---

In this next block we will build the attention mechanism. This is the key component of the transformer. To recap from the lectures, there are a few steps that we need to follow.


1.   We get three inputs passed into the attention heads, which then pass through the $W^Q$, $W^K$ and $W^V$ matrices, which all have dimensions $\left( d_\textrm{model}, d_k \right)$ to give us queries $Q$, keys $K$, and values $V$.
2.   Calculate $A = \textrm{softmax}\left( \frac{QK^T}{\sqrt{d_k}} \right)$
3.   The output from the attention head is given by the matrix product $AV$
4.   Repeat the above for each attention head and concatenate the output
5.   Pass the concatenated output through the dense layer representing matrix $W^0$ with shape $\left( \left(n_\textrm{heads}d_k \right) \times d_\textrm{model}\right)$

Implementation information:

*   We saw in the lectures how to represent the weight matrices as fully-connected layers (`torch.nn.Linear`), where the number of neurons is equal to the number of columns in the matrix. We need to ensure that we don't use a bias term and that we are not using an activation function.
* Use `torch.matmul(A,B)` to multiply matrix $A$ by matrix $B$. You can make use of the `torch.tensor` `transpose` function to get the transposed matrix: `B.transpose(-2,-1)`


In [ ]:
import math
import torch
import torch.nn as nn

# Custom attention layer
class CustomAttention(nn.Module):
    def __init__(self, d_model, d_k, num_heads):
        super().__init__()

        self.d_k = d_k
        self.num_heads = num_heads

        # Wq
        self.Wq = nn.ModuleList([
            nn.Linear(d_model, d_k, bias=False)
            for _ in range(num_heads)
        ])

        self.Wk = nn.ModuleList([
            nn.Linear(d_model, d_k, bias=False)
            for _ in range(num_heads)
        ])

        self.Wv = nn.ModuleList([
            nn.Linear(d_model, d_k, bias=False)
            for _ in range(num_heads)
        ])

        # Output projection
        self.W0 = nn.Linear(num_heads * d_k, d_model, bias=False)

        self.softmax = nn.Softmax(dim=-1)
        self.layernorm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(0.1)

    def forward(self, q, k, v, mask=None, return_attention=False):
        """
        q, k, v shape:
            (batch_size, seq_len, d_model)

        mask shape (typically):
            (batch_size, seq_len, seq_len)
        """

        Z = []
        attention_weights = []

        for head in range(self.num_heads):

            # Perform the projections
            Q = self.Wq[head](q)
            K = self.Wk[head](k)
            V = self.Wv[head](v)

            # QK^T
            QKT = torch.matmul(Q, K.transpose(-2, -1))

            # Apply the mask if we have one
            if mask is not None:
                QKT = QKT + mask

            # Scale
            QKT = QKT / math.sqrt(self.d_k)

            # Apply the softmax
            A = self.softmax(QKT)

            # This is just if you want to look at attention
            # weights like I showed in the lecture
            if return_attention:
                attention_weights.append(A.detach())

            # Apply the dropout
            A = self.dropout(A)

            # Multiply the attention by V
            att_output = torch.matmul(A, V)

            Z.append(att_output)

        # Concatenate the output from the different heads
        concZ = torch.cat(Z, dim=-1)

        # Pass this through the final linear layer W0
        output = self.W0(concZ)

        # Add more dropout at this stage
        output = self.dropout(output)

        # Residual connection + LayerNorm
        output = self.layernorm(q + output)

        if return_attention:
            return output, attention_weights

        return output



---

Since this is the most important part of the transformer, let's perform a quick test to see if we get the expected answer. If your attention layer is correct then you will see the following output:

```
torch.Size([1, 5, 4])
tensor([[[-0.5129,  0.0423, -1.1108,  1.5814],
         [-1.0987, -0.0903, -0.4247,  1.6137],
         [ 0.1606, -1.6418,  1.0420,  0.4392],
         [-0.0711, -1.3950,  1.4303,  0.0358],
         [ 0.3760, -0.7331, -1.0974,  1.4546]]])
```

As expected, the output from the layer has the same shape as the input.


In [ ]:
import torch

torch.manual_seed(42)

test_model = CustomAttention(
    d_model=4,
    d_k=3,
    num_heads=8
)

test_model.eval()

with torch.no_grad():
    unit_test_1_output = test_model(
        unit_test_1_input,
        unit_test_1_input,
        unit_test_1_input,
        mask=None
    )

print(unit_test_1_output.shape)
print(unit_test_1_output)

---

Now we create the feed forward network. This is a very simple network consisting of just two `Linear` layers. The first layer expands the dimensions of the input to `ff_dim` and the second contracts it back down to the size of the input, `d_model`. Here the original transformer uses a `relu` activation on the first layer, and no activation in the second.

In [ ]:
import torch
import torch.nn as nn

class FeedForward(nn.Module):
    def __init__(self, d_model, ff_dim):
        super().__init__()

        # Fully-connected layers
        self.fc_1 = nn.Linear(d_model, ff_dim)
        self.fc_2 = nn.Linear(ff_dim, d_model)

        # Layer normalisation
        self.layernorm = nn.LayerNorm(d_model)

        # Dropout
        self.dropout = nn.Dropout(0.1)

    def forward(self, inputs):
        # First fully-connected layer followed by dropout
        output = self.fc_1(inputs)
        output = torch.relu(output)
        output = self.dropout(output)

        # Second fully-connected layer followed by dropout
        output = self.fc_2(output)
        output = self.dropout(output)

        # Residual connection and normalisation
        output = self.layernorm(inputs + output)

        return output



---


Let's add a second little test. We'll pass the output of the attention through the feed forward layer and see what we get.

```
torch.Size([1, 5, 4])
tensor([[[-1.3388,  0.7243, -0.5568,  1.1713],
         [-0.7205,  0.7747,  1.1725, -1.2267],
         [ 0.3612, -0.7313,  1.4629, -1.0929],
         [ 0.7995,  0.3864, -1.7120,  0.5262],
         [-0.3613,  1.4937,  0.1398, -1.2722]]])
```
   Again, the output size will match the original input size, and the elements should be as above


In [ ]:
import torch

torch.manual_seed(42)

test_model_2 = FeedForward(
    d_model=4,
    ff_dim=20
)

test_model_2.eval()

with torch.no_grad():
    unit_test_2_output = test_model_2(unit_test_2_input)

print(unit_test_2_output.shape)
print(unit_test_2_output)

---

Now we have our building blocks, we just need to put them together to make the encoder and decoder. The encoder is now very simple - we just need to perform two steps:
1. Self-attention: The same input is used for the  `𝑞` ,  `𝑘`  and  `𝑣`  inputs to the `CustomAttention` layer we defined earlier. We need to make sure to pass on the `mask` here too.
2. The output from the self-attention goes through the `FeedForward(d_model, ff_dim)` layer.

Here you need to define the `self.attention` and `self.feedforward` class variables using the constructors of `CustomAttention` and `FeedForward` that we defined above, making sure to pass the necessary parameters to them, such as `d_model`.

Then in the `call` function we actually call the layers and apply them.

In [ ]:
import torch.nn as nn


class Encoder(nn.Module):
    def __init__(self, d_model, d_k, ff_dim, num_heads):
        super().__init__()

        # Self-attention
        self.attention = CustomAttention(d_model, d_k, num_heads)

        # Feed-forward network
        self.feedforward = FeedForward(d_model, ff_dim)

    def forward(self, inputs, mask=None):
        # Self-attention: q, k and v are all the same
        # for encoder self-attention
        z = self.attention(q=inputs, k=inputs, v=inputs, mask=mask)

        # Feed-forward network
        output = self.feedforward(z)

        return output

Let's test our encoder now. As before, we should get a tensor with the exact values:
```
torch.Size([1, 5, 4])
tensor([[[-0.6100, -1.2482,  0.5145,  1.3436],
         [ 0.0254, -1.6268,  0.9886,  0.6128],
         [ 0.9916, -1.1905, -0.7892,  0.9881],
         [-1.6757,  0.5941,  0.1843,  0.8973],
         [-0.0183, -0.7874,  1.6388, -0.8331]]])
```

In [ ]:
torch.manual_seed(42)

test_model_3 = Encoder(
    d_model=4,
    d_k=3,
    ff_dim=20,
    num_heads=8
)

test_model_3.eval()

with torch.no_grad():
    unit_test_3_output = test_model_3(
        unit_test_3_input,
        mask=None
    )

print(unit_test_3_output.shape)
print(unit_test_3_output)

---

The decoder is marginally more complex than the encoder, since we need to have two attention layers:
1. Masked self-attention: input to the decoder is used for the `𝑞`, `𝑘` and `𝑣` inputs to the `CustomAttention` layer we defined earlier. We need to make sure we are using the mask for the decoder input here.
2. Cross-attention: this uses the output of the encoder as the $k$ and $v$ inputs, but the output of the masked attention as $q$. We use the `CustomAttention` layer again here, making sure to use the mask that we created for the encoder input.
3. The output is passed into a feed-forward network to give the final ouput of the decoder, using the `FeedForward` layer.


In [ ]:
import torch.nn as nn


class Decoder(nn.Module):
    def __init__(self, d_model, d_k, ff_dim, num_heads):
        super().__init__()

        # Masked self-attention
        self.masked_attention = CustomAttention(
            d_model, d_k, num_heads)

        # Cross-attention
        self.cross_attention = CustomAttention(
            d_model, d_k, num_heads)

        # Feed-forward network
        self.feedforward = FeedForward(d_model, ff_dim)

    def forward(
        self,
        inputs,
        encoder_output,
        decoder_mask=None,
        encoder_mask=None
    ):

        # Masked self-attention
        z = self.masked_attention(
            q=inputs,
            k=inputs,
            v=inputs,
            mask=decoder_mask
        )

        # Encoder-decoder cross-attention
        c = self.cross_attention(
            q=z,
            k=encoder_output,
            v=encoder_output,
            mask=encoder_mask
        )

        # Feed-forward network
        output = self.feedforward(c)

        return output

A final unit test. Let's make sure that the decoder is working correctly. We'll use the output from the test of the encoder and a random input.
```
torch.Size([1, 5, 4])
tensor([[[-0.2760, -1.0396,  1.6522, -0.3366],
         [-0.2327, -0.2098,  1.6004, -1.1578],
         [-1.2905, -0.5022,  1.3840,  0.4087],
         [ 1.7184, -0.7776, -0.4699, -0.4709],
         [-1.7206,  0.4160,  0.5647,  0.7400]]])
```

In [ ]:
torch.manual_seed(42)

test_model_4 = Decoder(
    d_model=4,
    d_k=3,
    ff_dim=20,
    num_heads=8
)

test_model_4.eval()

with torch.no_grad():
    unit_test_4_output = test_model_4(
        unit_test_4_input,
        unit_test_3_output,
        decoder_mask=None,
        encoder_mask=None
    )

print(unit_test_4_output.shape)
print(unit_test_4_output)

---

Now all we need to do is put the building blocks together to create the full transformer model.
1. Create the three masks that we need to make use of:
> 1.   Padding mask for the encoder input
> 2.   Padding mask for the decoder input
> 3.   Causal mask for the decoder (prevents us looking into the future when predicting the output)
2. Perform the sentence embedding for encoder (English) and decoder (Spanish) inputs
3. Call the encoder with the English input and mask
4. Call the decoder with the Spanish input, theoutput from the encoder, the combination of the decoder masks and the encoder mask
5. Make the predictions from the output of the decoder
6. Sit back and enjoy some translations soon

In [ ]:
import torch
import torch.nn as nn


class Transformer(nn.Module):
    def __init__(
        self,
        sequence_length,
        d_model,
        d_k,
        ff_dim,
        num_heads,
        vocab_size_english,
        vocab_size_spanish,
        learned_encoding
    ):
        super().__init__()

        # Encoder and decoder sequence lengths
        self.l_e = sequence_length
        self.l_d = sequence_length + 1

        # Embedding and positional encoding
        self.english_embed = EmbedAndEncode(
            self.l_e, d_model, vocab_size_english, learned_encoding)

        self.spanish_embed = EmbedAndEncode(
            self.l_d, d_model, vocab_size_spanish, learned_encoding)

        # Encoder and decoder
        self.encoder = Encoder(d_model, d_k, ff_dim, num_heads)

        self.decoder = Decoder(d_model, d_k, ff_dim, num_heads)

        self.dropout = nn.Dropout(0.1)

        # Final classifier layer
        self.classifier = nn.Linear(d_model, vocab_size_spanish)

    def forward(self, enc_in, dec_in):

        # Masks
        enc_padding_mask = self.create_padding_mask(enc_in)
        dec_padding_mask = self.create_padding_mask(dec_in)
        dec_causal_mask = self.create_causal_mask(dec_in.device)

        # Combine decoder masks
        dec_mask = torch.minimum(dec_padding_mask, dec_causal_mask)

        # Encoder
        enc_in = self.english_embed(enc_in)
        enc_in = self.dropout(enc_in)
        enc_out = self.encoder(enc_in, enc_padding_mask)

        # Decoder
        dec_in = self.spanish_embed(dec_in)
        dec_in = self.dropout(dec_in)
        dec_out = self.decoder(dec_in, enc_out, dec_mask, enc_padding_mask)

        # Classification: apply dropout and then the final layer
        dec_out = self.dropout(dec_out)
        logits = self.classifier(dec_out)

        # In torch the softmax is embedded in the loss function that
        # we will use, hence we return the logits before softmax here
        return logits

    # Create the causal mask - effectively a triangular matrix
    def create_causal_mask(self, device):
        mask = torch.triu(
            torch.ones(
                self.l_d,
                self.l_d,
                device=device
            ) * -1.0e20,
            diagonal=1
        )

        return mask

    # Mask to prevent attending to padded values
    def create_padding_mask(self, inputs):

        mask = (inputs == 0).float()

        # Shape:
        # (batch_size, 1, seq_len)
        mask = mask.unsqueeze(1)

        mask *= -1.0e20

        return mask

---

Now lets define the parameters of the model and build it. To keep things reasonably sized we'll create a fairly small network. At some point you could try changing these if you have a GPU that you can use for training, but you will need to use these values to load my model weights later.

*   $d_\textrm{model} = 256$
*   $n_\textrm{heads} = 8$
*   $d_k = 32$
*   $\textrm{ff_dim} = 1024$

In this example I have set $d_k = d_v$, so we won't see $d_v$ in this code. This follows the method used in the original transformer, but you could modify the layers above to allow for a different value of $d_v$

Using these values, you should find that the network has `13385624` parameters


In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")

# Define the model hyper-parameters
d_model = 256
num_heads = 8
d_k = 32
ff_dim = 1024

# Create the transformer model
transformer = Transformer(
    sequence_length=sequence_length,
    d_model=d_model,
    d_k=d_k,
    ff_dim=ff_dim,
    num_heads=num_heads,
    vocab_size_english=vocab_size_english,
    vocab_size_spanish=vocab_size_spanish,
    learned_encoding=True
)

# Print a summary of the model (can be quite verbose)
import torchinfo
enc_in, dec_in, labels = training_dataset[0]
print((1,*enc_in.shape), (1,*dec_in.shape))
torchinfo.summary(transformer, input_data=(enc_in.unsqueeze(0), dec_in.unsqueeze(0)))


In [ ]:
import torch.optim as optim

# Use the optimiser settings from the original paper (but not the lr mechanism)
optimiser = optim.Adam(
    transformer.parameters(),
    lr=0.001,
    betas=(0.9, 0.98),
    eps=1.0e-9
)

# We can use a learning rate scheduler to reduce the learning rate if necessary
# Not really necessary here as it will take quite a few epochs to be useful,
# but I leave it here for completeness
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimiser,
    patience=3
)

# We ignore the padding class in the loss function and smooth the labels a little
criterion = torch.nn.CrossEntropyLoss(
    ignore_index=0,
    label_smoothing=0.1
)

---

Now we can train the network! Unfortunately this will be rather slow so in the next block you can load some weights from a network I trained previously. On a V100 it trains very quickly - less than 1 minute per epoch! If you have a GPU available, go ahead and train it for 10 epochs or so, otherwise feel free to skip this block


In [ ]:
num_epochs = 15

transformer.to(device)

for epoch in range(num_epochs):

    # Training
    transformer.train()

    running_loss = 0.0

    for encoder_input, decoder_input, target in training_dataloader:

        encoder_input = encoder_input.to(device)
        decoder_input = decoder_input.to(device)
        target = target.to(device)

        optimiser.zero_grad()

        logits = transformer(
            encoder_input,
            decoder_input
        )

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            target.reshape(-1)
        )

        loss.backward()

        optimiser.step()

        running_loss += loss.item()

    train_loss = running_loss / len(training_dataloader)

    # Validation
    transformer.eval()

    validation_loss = 0.0

    with torch.no_grad():

        for encoder_input, decoder_input, target in validation_dataloader:

            encoder_input = encoder_input.to(device)
            decoder_input = decoder_input.to(device)
            target = target.to(device)

            logits = transformer(
                encoder_input,
                decoder_input
            )

            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                target.reshape(-1)
            )

            validation_loss += loss.item()

    validation_loss /= len(validation_dataloader)

    scheduler.step(validation_loss)

    current_lr = optimiser.param_groups[0]["lr"]

    print(
        f"Epoch {epoch+1}/{num_epochs} "
        f"Train Loss: {train_loss:.4f} "
        f"Val Loss: {validation_loss:.4f} "
        f"LR: {current_lr:.6f}"
    )

# Save the weights if you want to keep them
torch.save(
    transformer.state_dict(),
    "my_transformer_weights.pt"
)

---

Training is unfortunately very slow on a CPU, so if you don't have any GPU access then feel free to load some model weights that I produced earlier


In [ ]:
# Download the PyTorch weights
!wget --no-check-certificate \
'https://www.hep.phy.cam.ac.uk/~lwhitehead/lhw_weights.pt' \
-O lhw_weights.pt

transformer.load_state_dict(
    torch.load(
        "lhw_weights.pt",
        map_location=device
    )
)


---

Now all we need to do is look at how to perform inference. As I said in the lectures, this is done in an iterative way. We pass the english sentence into the encoder, and then pass the `<sos>` (start of sentence) token to the decoder. The decoder then predicts the output word, which we then append to the `<sos>` token and run things again with this new input.

In [ ]:

# Get our vocabulary from the vectorisation layer we made right back at the top
# of the notebook
index_to_spanish = {
    index: word
    for word, index in spanish_vocab.items()
}

import torch

def run_inference(input_sentence):

    transformer.eval()

    with torch.no_grad():

        # Encode English input
        encoded_input = vectorise_sentence(
            input_sentence,
            english_vocab,
            sequence_length
        )

        encoded_input = torch.tensor(
            encoded_input,
            device=device,
            dtype=torch.long
        ).unsqueeze(0)  # add batch dimension

        # Decoder starts with <sos>
        decoded_sentence = ["<sos>"]

        for i in range(sequence_length):

            decoder_text = " ".join(decoded_sentence)

            decoder_input = vectorise_sentence(
                decoder_text,
                spanish_vocab,
                sequence_length + 1
            )

            decoder_input = torch.tensor(
                decoder_input,
                device=device,
                dtype=torch.long
            ).unsqueeze(0)


            # Forward pass
            predictions = transformer(encoded_input, decoder_input)

            # Select the prediction for this timestep
            predicted_word_index = torch.argmax(predictions[0, i, :]).item()
            predicted_word = index_to_spanish[predicted_word_index]

            decoded_sentence.append(predicted_word)


            if predicted_word == "<eos>":
                break


    # Remove special tokens
    output = " ".join(decoded_sentence)
    output = (output.replace("<sos> ", "").replace(" <eos>", ""))

    print(input_sentence, " :: ", output)

Now for the fun part! I've added a few sentences for the transformer to try to translate. The first sentence is, of course, the one from the lectures. Feel free to play around and add your own sentences and see how it does (if you can't speak spanish, you could ask Lorena, Google, an LLM, and in a crisis, me.)

If you've loaded my weights, then the five sentences below are translated correctly. As a bit of a language technicality, in the lectures I said that `I have a big cat` should translate to `Yo tengo un gato grande`. You may notice that `Yo`, meaning `I`, is missing - pronouns such as `yo` are often dropped from spanish since the conjugation of the verb (`tengo`) tells you that the pronoun is `yo` anyway.

In [ ]:
run_inference("i have a big cat")
run_inference("i have a small dog")
run_inference("are there horses here")
run_inference("my car is broken")
run_inference("im going to translate this sentence")